# Optional Exercise - Pursue if time allows
# Exercise 4 – Jupyter Notebook
# Build a service ticket intelligence classification model using AutoML and raw text features
SAP TechEd 2026, Hands-On Workshop: DA262 - Get hands-on with tabular AI- and forecasting-models in SAP HANA Cloud
<br><br>



## Understanding the exercise scenario 

This exercise builds upon a previous years SAP TechEd Hands-on Workshop [github.com/SAP-archive/teched2020-INT260](https://github.com/SAP-archive/teched2020-INT260) around a __Service Ticket Intelligence__-use case on __travel agency service tickets__ [ex2-STI](https://github.com/SAP-archive/teched2020-INT260/blob/main/exercises/ex2-STI/ex2.2/README.md), resumed here leveraging capabilities purely out of SAP HANA Cloud __AI function library (AutoML with the Predictive Analysis Library PAL)__ and __Text Embedding Models from the NLP services__ in SAP HANA Cloud.

Overview on the exercise tasks

![](./images/ex4_scenario_overview.png)

<br>

## Ex. 4.0 - Connect to your SAP HANA Cloud instance

### Step 0: Establish and check connection


Throughout this exercise, we will be using the __Python Machine Learning client for SAP HANA (hana-ml)__, as your reference to all its functions and capabilities see the [hana-ml documentation](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/change_log.html). The current version released with SAP HANA Cloud 2026 Q3 release is 2.30.
- The python package hana-ml in general allows to script in python, while SQL code is generated on-the-fly and directly passed to a connected SAP HANA database system for execution.
    - It allows to access and prepare data by means of a HANA dataframe, a python object holding a SQL select query. Many methods are provided to be used with the HANA dataframe, changing the SQL select query behind the scenes.
    - As its core, it provides methods to apply AI functions (algorithms from the [Predictive Analysis Library (PAL)](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-predictive-analysis-library/sap-hana-cloud-sap-hana-database-predictive-analysis-library-pal?locale=en-US&version=LATEST) and [Automated Predictive Library (APL)](https://help.sap.com/docs/apl?locale=en-US&version=LATEST) to the data prepared with a HANA dataframe, designed to apply all the processing within the SAP HANA database.
- The python package is released as a component with every the SAP HANA Client delivery, in addition the latest __hana-ml__-version can always be found at the pypi public repository at https://pypi.org/project/hana-ml/.
<br><br>

In Python, installed packages require to be __imported__ to the current session, in order to be available for use in python scripts.
- Let's first run the import of hana-ml

In [ ]:
# Importing the Python Machine Learning client library for SAP HANA and get the version
import hana_ml
print(hana_ml.__version__)

Next, execute the next cell and the referenced script, prepared in the Getting Started section to connect to the SAP HANA CLoud database.

In [ ]:
%run "../ex0/ex0_connect2HANACloud.ipynb"

<br><br>

## Ex. 4.1 - Exploring Travel Service Ticket data using SAP HANA Dataframes

### Step 1: Load and explore the travel service ticket data
The datafile is already downloaded to your local workspace from the github repository [teched2020-INT260/ex2-STI/ex2.2](https://github.com/SAP-archive/teched2020-INT260/blob/main/exercises/ex2-STI/ex2.2/README.md)

In [ ]:
#df = pd.read_csv("../datasets/travel.csv")
import pandas as pd

import hana_ml.dataframe as dataframe
travel_sti_hdf = dataframe.create_dataframe_from_pandas(myconn ,
                                            pd.read_csv("./travel.csv"),
                                            table_name="TRAVEL_SERVICE_TICKETS",
                                            force=True
                                            # ,table_structure={ "<column name>" : "<data type>",  "<column name>" : "<data type>", ...}         
                                            )

In [ ]:
# Adding an ID column to the dataframe
travel_sti_hdf=travel_sti_hdf.add_id()

# Sampling 10 travel service ticket rows for exploring some sample rows
from hana_ml.algorithms.pal.preprocessing import Sampling
smp = Sampling(method='simple_random_without_replacement', sampling_size=10)
sample_tickets = smp.fit_transform(data=travel_sti_hdf)

# Displaying the rows
pd.set_option('display.max_colwidth', None)
display(sample_tickets.collect().style.hide())

<br> 

__Explore the ticket category distribution__
- After loading the data into dataframe, we check the distribution of classes (target variable).
- Usually the model works best if the dataset is balanced i.e. classes are equally distributed with little skewness, and each class has at least 1000 data points.

In [ ]:
display(travel_sti_hdf.agg([('count', 'Description', 'COUNT')], group_by='Category').collect())

<br><br>

## Ex 4.2 - Building a AutoML classification model for Travel Service Tickets

### Step 2: Training, test data split for evaluating trained model

We split the loaded data into two subsets
- The core subset for __training__ the model
- The __hold-out subset__ we use for model evaluation/testing. It is called hold_out, as the training task has never seen the data.

In [ ]:
# Split the data into a model training- and a houl-out sample for testing the classification model
# Stratified ensures, Categories are evenly distributed in both subsets
from hana_ml.algorithms.pal.partition import train_test_val_split
travel_sti_hdf_train, travel_sti_hdf_test, empty_val_hdf = train_test_val_split(data=travel_sti_hdf.select('ID', 'Description', 'Category'), partition_method='stratified', stratified_column='Category',
                                                       training_percentage=0.85,
                                                       testing_percentage=0.15,
                                                       validation_percentage=0.0)

print('Training data sample shape: ', travel_sti_hdf_train.shape)
print('The HANA dataframe result set shape: ',travel_sti_hdf_train.columns)
print('Testing, hold-out data sample shape: ',travel_sti_hdf_test.shape)

display(travel_sti_hdf_train.head(1).collect())
display(travel_sti_hdf_train.agg([('count', 'Description', 'COUNT')], group_by='Category').collect())
display(travel_sti_hdf_test.agg([('count', 'Description', 'COUNT')], group_by='Category').collect())

<br>

Briefly save the HANA dataframe to TEMPORARY database tables

In [ ]:
travel_sti_hdf_train.save("#TRAVEL_SERVICE_TICKETS_TRAIN", force=True)
travel_sti_hdf_test.save("#TRAVEL_SERVICE_TICKETS_TEST", force=True)
#itst_hdf_val.save("#TRAVEL_SERVICE_TICKETS_TRAIN_MINI", force=True)

<br>

### Step 3: Building a AutoML classification model for IT Service Tickets

The __target__ column, we want the model to predict is __'Category'__  
The only __feature column__ we want to feed into the model is the raw text column __'Description'__.

In [ ]:
class_features=['Description']

<br>

Now, let's create a SAP HANA Cloud Predictive Analysis Library (PAL) __AutoML classifier__ model
- __first__, we create our AutomaticClassification object instance __'automl_class'__
- with some global AutoML scenarios settings

In [ ]:
# Creating the PAL AutoML classification scenario and general settings 
from hana_ml.algorithms.pal.auto_ml import AutomaticClassification

automl_class = AutomaticClassification(config_dict='default',      #'default', 'light', <custom> AutoML scenario configuration (selection of algorithms and paramters)  
                                       search_method='GA', population_size=25, generations=3,  offspring_size=15,  # GA: genetic algorithm for optimization
                                       successive_halving=False,                                                   # Use successive halving for faster convergence to the optimal pipeline
                                       min_layer=2,                                                                # The pipeline model must have 2 operators in sequence as minimum
                                       random_seed=20261028,
                                       fold_num=3, resampling_method='stratified_cv',                              # Cross-validation settings
                                       scorings={"ACCURACY": 1.0, "AUC": 1.0},                                     # Model accuracy metrics used for judging what we optimize for
                                       elite_number=15                                                             # Number of best models we want to review afterwards
                                         )    

<br>

A fundamental, and crucial configuration in AutoML is, __which algorithms (operators)__ and __parameter values (ranges)__ we want to define for the AutoML task.  
That sets the __space and bounderies__ for the __AutoML fit task to explore__.  
- __Secondly__, that's what we define with the __config_dict__ which we adjust now

In [ ]:
# Display the classifier algorithms and their configurations included in the AutoML task
#automl_class.display_config_dict(category="Classifier")

# For more targeted AutoML models, we strip down the AutoML operators to focus on the TextEmbedding- and some classification-operators only
# Drop all Resampler-operators
automl_class.delete_config_dict(category='Resampler')

# Drop selected Transformer-operators
automl_class.delete_config_dict('TargetEncoder') 
automl_class.delete_config_dict('LabelEncoder')
automl_class.delete_config_dict('OneHotEncoder')
automl_class.delete_config_dict('PolynomialFeatures')
automl_class.delete_config_dict('CATPCA')
automl_class.delete_config_dict('Imputer')
automl_class.delete_config_dict('SCALE')
automl_class.delete_config_dict('FS_unsupervised')
automl_class.delete_config_dict('FS_supervised')

# Drop selected Classifier-operators
automl_class.delete_config_dict('M_LOGR_Classifier')
automl_class.delete_config_dict('NB_Classifier')
automl_class.delete_config_dict('SVM_Classifier')
automl_class.delete_config_dict('DT_Classifier')

# Adjusting the TextEmbedding-operator parameters
automl_class.update_config_dict('TextEmbedding', param_name='MODEL_VERSION', param_config=['SAP_GXY.20250407'])
automl_class.update_config_dict('TextEmbedding', param_name='PCA_DIM_NUM', param_config=[64, 128, 256])

# Adjusting the TFIDF-operator parameters
automl_class.update_config_dict('TFIDF', param_name='EMBED_DIM', param_config=[64, 128, 256])

# Display final AutoML configuration
print(automl_class.display_config_dict())

<br>

The __AutoML-fit task__ is started by executing the next cell
- the search __progress can be visually observed__ with the __PipelineProgressStatusMonitor__
- note, with the fit-method, we specifically set __text_variable__-parameter to handle our only feature the __'Description'__ column as a raw text column, requiring some preprocessing
- further note, the __executing of the AutoML fit task__, depending on system load, __may take 3-4 minutes (~230 seconds)__, be patient and observe the AutoML progress monitor.

In [ ]:
# Start the AutoML process and visualize the progress in evaluating and finding a best possible classification model
from hana_ml.visualizers.automl_progress import PipelineProgressStatusMonitor
progress_status_monitor = PipelineProgressStatusMonitor(connection_context=myconn, automatic_obj=automl_class, runtime_platform='vscode')
progress_status_monitor.start()

automl_class.enable_workload_class("DA262_WLC")                 # A SAP HANA workload class is applied to restrict resource usage by AutoML on the system
automl_class.fit(data=travel_sti_hdf_train, 
                 key='ID', 
                 features=class_features, 
                 text_variable=class_features,                  # TEXT_VARIABLE: process column(s) as plain text input column, apply preprocessing for semantically unlocking the text
                 label='Category')   

automl_class.runtime                                            # Depending on system configuration, runtime maybe be in ranage of ~102-200secs secs on a 3 vCPUs instance

In [ ]:
# Review the execute SQL for the AutoML-fit task
print(automl_class.get_fit_execute_statement())


<br>

__Now, review the ELITE (best) Pipelines identified by the AutoML-fit task and their accuracy__
- Which classifier-algorithms were used?
- Which text preprocessing-operators and parameter were used?
- Which is the best pipeline model? The first one with ID 0

In [ ]:
automl_class.best_pipeline_.collect()

In [ ]:
#display(automl_class.get_best_pipeline())

<br><br>

## Ex 4.3 - Evaluate the AutoML model accuracy

### Step 4: Apply the trained model using the score-function to the hold-out test data

<br>

Apply the __score()-method__ with the __hold-out sample__, to __evaluate the model's "performance" (i.e. accuracy etc.)__ on data unseen during fitting of the model
- we use the score-method of our automl_class-object
- we use the travel_sti_hdf_test-dataframe

In [ ]:
score_pred_hdf, score_stats_hdf = automl_class.score( data=travel_sti_hdf_test, key='ID', features=class_features, label='Category')
automl_class.runtime 

In [ ]:
# In case we want to review the executed SQL statement for the score method using the PAL_PIPELINE_SCORE-SQL function
#print(automl_class.get_score_execute_statement())

<br>

Review the model accuracy statistics applying the hold-out subset

In [ ]:
# Review the statistics - e.g. Accuracy and AUC
display(score_pred_hdf.select('ID', 'SCORE', 'CONFIDENCE').head(3).collect())
display(score_stats_hdf.collect())

<br>

Visualize the classification __confusion matrix__ for the best AutoML pipeline model applied to the hold-out sample

In [ ]:
# Prepare a dataframe with incl. actual Ticket class categories and the predicted Ticket class categories
itst_score_mtrx_hdf=travel_sti_hdf_test.set_index("ID").join(score_pred_hdf.set_index("ID"))
itst_score_mtrx_hdf=itst_score_mtrx_hdf.rename_columns({'SCORE': 'PREDICTED_CLASS', 'Category': 'ACTUAL_CLASS'}).select('ID', 'ACTUAL_CLASS', 'PREDICTED_CLASS')

# Build the confusion matrix
from hana_ml.algorithms.pal import metrics
cm_df, classification_report_df = metrics.confusion_matrix(itst_score_mtrx_hdf, key='ID', label_true='ACTUAL_CLASS', label_pred='PREDICTED_CLASS')

# Plot the confustion matrix
from hana_ml.visualizers.metrics import MetricsVisualizer
cm_viz = MetricsVisualizer(show_plotly=False)
cm_viz.plot_confusion_matrix(df=cm_df, normalize=False) #normalize=False|True)

<br>

## Ex.4.4 - Predicting with the AutoML classification model

### Step 5: Applying the AutoML classification model to new travel tickets for predictions

In [ ]:
# Apply the automl_class-model using the predict-method
hdf_predictions = automl_class.predict( data=travel_sti_hdf_test.drop('Category'), key='ID', features=class_features)

# Joining the prediction results with the original ticket-text column
hdf_result=travel_sti_hdf_test.select('ID', 'Description').set_index("ID").join(hdf_predictions.set_index("ID"))
hdf_result=hdf_result.rename_columns({'SCORES': 'PREDICTED_TRAVEL_TICKET_CATEGORY'}).sort("ID")

# Sample 10 IDs from the predicted results
from hana_ml.algorithms.pal.preprocessing import Sampling
smp = Sampling(method='simple_random_without_replacement', sampling_size=10)
sample_ticket_predictions = smp.fit_transform(data=hdf_result)
display(sample_ticket_predictions.collect().style.hide())

In [ ]:
# In case we want to review the executed SQL statement for the predict method using the PAL_PIPELINE_PREDICT-SQL function
#print(automl_class.get_predict_execute_statement())